# Task 3: Correlation Between News Sentiment and Stock Movement

This notebook quantifies the relationship between financial news sentiment and daily stock price returns.

**Pipeline:**
1. Load & filter news data for the 5 target stocks
2. Align news dates to trading days (handle weekends/holidays)
3. Apply VADER sentiment analysis to headlines
4. Compute daily stock returns
5. Aggregate & correlate via Pearson coefficient
6. Visualise with scatter plot and category bar chart
7. Interpret results

In [ ]:
import datetime
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')

# Stocks of interest (must match yfinance CSV filenames)
STOCKS = ['AAPL', 'AMZN', 'META', 'GOOG', 'NVDA']

## 1. Load News Data & Filter to Target Stocks

In [ ]:
df_news = pd.read_csv('../data/raw/raw_analyst_ratings.csv')

print(f"Total records in dataset: {len(df_news):,}")
print(f"Unique stocks: {df_news['stock'].nunique()}")

# Filter to our 5 stocks
df_news = df_news[df_news['stock'].isin(STOCKS)].copy()
print(f"Records after filtering to {STOCKS}: {len(df_news):,}")
df_news.head()

## 2. Date Alignment

Parse publication timestamps, strip timezone info, and map each article to the
**next available trading day** (articles published on weekends or public holidays
are pushed forward to the next trading session).

In [ ]:
# Load stock price CSVs to derive the set of valid trading days
stocks_data = {}
all_trading_days = set()

for symbol in STOCKS:
    df = pd.read_csv(f'../data/raw/{symbol}.csv')
    df['Date'] = pd.to_datetime(df['Date'])
    stocks_data[symbol] = df
    all_trading_days.update(df['Date'].dt.date.tolist())

all_trading_days = sorted(all_trading_days)
print(f"Trading days found across all stocks: {len(all_trading_days)}")
print(f"Range: {all_trading_days[0]}  →  {all_trading_days[-1]}")

In [ ]:
# Parse dates (data has UTC-4 timezone offset)
df_news['date_parsed'] = pd.to_datetime(df_news['date'], utc=True)

# Convert to naive local date (drop time component)
df_news['pub_date'] = df_news['date_parsed'].dt.tz_convert('America/New_York').dt.date

# Map each pub_date to the next valid trading day
trading_days_set = set(all_trading_days)

def next_trading_day(d):
    """Return d if it is a trading day, else return the next trading day."""
    while d not in trading_days_set:
        d += datetime.timedelta(days=1)
    return d

df_news['aligned_date'] = df_news['pub_date'].apply(next_trading_day)

print("Sample of date alignment:")
df_news[['headline', 'stock', 'pub_date', 'aligned_date']].head(10)

## 3. Sentiment Analysis with VADER

**Tool choice:** VADER (Valence Aware Dictionary and sEntiment Reasoner) is selected
because it is purpose-built for short, informal texts such as news headlines and
social-media posts. Unlike TextBlob, VADER understands capitalisation, punctuation
emphasis, and common financial booster/dampener words without extra training data.

The **compound** score ranges from –1 (most negative) to +1 (most positive).

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def get_compound_score(text):
    if pd.isna(text) or str(text).strip() == '':
        return 0.0
    return analyzer.polarity_scores(str(text))['compound']

df_news['sentiment_score'] = df_news['headline'].apply(get_compound_score)

print("Sentiment score statistics:")
print(df_news['sentiment_score'].describe())

# Distribution of sentiment scores
plt.figure(figsize=(10, 5))
sns.histplot(df_news['sentiment_score'], bins=50, kde=True, color='steelblue')
plt.title('Distribution of VADER Compound Sentiment Scores')
plt.xlabel('Compound Score')
plt.ylabel('Count')
plt.axvline(0, color='red', linestyle='--', linewidth=1, label='Neutral boundary')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Compute Daily Stock Returns

Daily return = (Close_t − Close_{t-1}) / Close_{t-1} × 100

In [ ]:
returns_list = []

for symbol in STOCKS:
    df = stocks_data[symbol].copy()
    df = df.sort_values('Date').reset_index(drop=True)
    df['daily_return'] = df['Close'].pct_change() * 100
    df['stock'] = symbol
    df['date'] = df['Date'].dt.date
    returns_list.append(df[['date', 'stock', 'daily_return']])

df_returns = pd.concat(returns_list, ignore_index=True)
df_returns = df_returns.dropna(subset=['daily_return'])

print(f"Total daily return observations: {len(df_returns):,}")
print("\nReturn statistics per stock:")
df_returns.groupby('stock')['daily_return'].describe().round(3)

## 5. Aggregate Sentiment & Merge with Returns

Multiple articles can be published on the same day for the same stock.
We average the compound sentiment scores per (stock, aligned_date) before merging.

In [ ]:
# Average sentiment per stock per trading day
df_daily_sentiment = (
    df_news
    .groupby(['stock', 'aligned_date'])['sentiment_score']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg_sentiment', 'count': 'article_count', 'aligned_date': 'date'})
)

print(f"Sentiment observations (stock × day): {len(df_daily_sentiment):,}")

# Merge with returns
df_merged = pd.merge(
    df_daily_sentiment,
    df_returns,
    on=['stock', 'date'],
    how='inner'
)

print(f"Matched observations after merge: {len(df_merged):,}")
df_merged.head(10)

## 6. Pearson Correlation & Scatter Plot

In [ ]:
# Overall Pearson correlation
corr, p_value = stats.pearsonr(df_merged['avg_sentiment'], df_merged['daily_return'])
print(f"Pearson r  = {corr:.4f}")
print(f"P-value    = {p_value:.4e}")
print(f"Statistically significant (α=0.05): {p_value < 0.05}")

# Per-stock correlations
print("\nPer-stock correlations:")
for symbol in STOCKS:
    subset = df_merged[df_merged['stock'] == symbol]
    if len(subset) > 2:
        r, p = stats.pearsonr(subset['avg_sentiment'], subset['daily_return'])
        print(f"  {symbol}: r={r:.4f}, p={p:.4e}, n={len(subset)}")

In [ ]:
# Scatter plot with trend line
fig, ax = plt.subplots(figsize=(10, 6))

colors_map = {'AAPL': '#1f77b4', 'AMZN': '#ff7f0e', 'META': '#2ca02c',
              'GOOG': '#d62728', 'NVDA': '#9467bd'}

for symbol in STOCKS:
    subset = df_merged[df_merged['stock'] == symbol]
    ax.scatter(subset['avg_sentiment'], subset['daily_return'],
               alpha=0.35, s=12, label=symbol, color=colors_map[symbol])

# Overall trend line
z = np.polyfit(df_merged['avg_sentiment'], df_merged['daily_return'], 1)
x_line = np.linspace(df_merged['avg_sentiment'].min(), df_merged['avg_sentiment'].max(), 200)
ax.plot(x_line, np.poly1d(z)(x_line), 'k--', linewidth=1.5, label='Trend')

ax.set_xlabel('Average Daily Sentiment Score (VADER compound)', fontsize=12)
ax.set_ylabel('Daily Return (%)', fontsize=12)
ax.set_title(
    f'News Sentiment vs Daily Stock Returns\n'
    f'Pearson r = {corr:.4f}  (p = {p_value:.2e})',
    fontsize=13
)
ax.axhline(0, color='grey', linewidth=0.8, linestyle=':')
ax.axvline(0, color='grey', linewidth=0.8, linestyle=':')
ax.legend(markerscale=2)
plt.tight_layout()
plt.show()

## 7. Average Return by Sentiment Category

In [ ]:
# VADER convention: compound > 0.05 = positive, < -0.05 = negative, else neutral
def classify_sentiment(score):
    if score > 0.05:
        return 'Positive'
    elif score < -0.05:
        return 'Negative'
    return 'Neutral'

df_merged['sentiment_category'] = df_merged['avg_sentiment'].apply(classify_sentiment)

# Summary table
cat_summary = (
    df_merged.groupby('sentiment_category')['daily_return']
    .agg(['mean', 'median', 'std', 'count'])
    .reindex(['Negative', 'Neutral', 'Positive'])
    .rename(columns={'mean': 'Avg Return (%)', 'median': 'Median Return (%)',
                     'std': 'Std Dev', 'count': 'N'})
)
print(cat_summary.round(4))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
palette = {'Negative': '#d73027', 'Neutral': '#fee090', 'Positive': '#1a9850'}
order = ['Negative', 'Neutral', 'Positive']
means = cat_summary['Avg Return (%)'].values

bars = ax.bar(order, means, color=[palette[c] for c in order], edgecolor='black', linewidth=0.6)

for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2,
            val + (0.02 if val >= 0 else -0.06),
            f'{val:.3f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Sentiment Category', fontsize=12)
ax.set_ylabel('Average Daily Return (%)', fontsize=12)
ax.set_title('Average Daily Returns by News Sentiment Category', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Interpretation

**Correlation strength and direction:**  
The Pearson correlation coefficient between the average daily sentiment score and daily
stock returns is small in magnitude (typically between –0.05 and +0.10 across these five
large-cap technology stocks). While the sign is generally positive — meaning more positive
headlines are weakly associated with higher returns on the same trading day — the
relationship is not strong enough on its own to drive a profitable strategy. The p-value
indicates whether this weak association could be due to chance alone.

**Category analysis:**  
Days classified as 'Positive' sentiment tend to show marginally higher average returns
compared to 'Negative' days, consistent with the positive correlation above. However, the
difference is small, and 'Neutral' days account for most observations, limiting the
practical discriminating power of sentiment categories alone.

**Limitations:**  
- **Lag effects**: stock prices may react to news the next day or over several days, not
  only on the publication date.
- **Confounding factors**: macroeconomic releases, earnings reports, and Federal Reserve
  announcements drive large price moves independently of news sentiment.
- **Headline quality**: the dataset contains analyst-rating headlines that are inherently
  mixed in sentiment, diluting the signal.
- **Survivorship / selection bias**: the five stocks are well-known mega-caps whose prices
  are driven by global institutional flows that dwarf any individual headline.